<a href="https://colab.research.google.com/github/yashwanthraaj1207-ops/Smart-Digital-Guardian/blob/main/Cyber_Club.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit pyngrok scikit-learn pandas

In [ ]:
%%writefile app.py

import streamlit as st
import re
import numpy as np
import pandas as pd
import os
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


# Creates the history

FILE = "search_history.csv"

if not os.path.exists(FILE):
    df = pd.DataFrame(columns=["Time", "Input", "Threat Score", "Result"])
    df.to_csv(FILE, index=False)


# Training the dataset

texts = [

    # ---------------- PHISHING EXAMPLES ----------------

    "free gift click now",
    "win money urgent claim",
    "verify your bank password",
    "click this phishing link",
    "claim your reward now",
    "urgent account verification required",
    "login to verify your account",
    "bank account suspended verify now",
    "free recharge click here",
    "google-login-security-alert.xyz",

    "your account has been hacked click here",
    "urgent action required verify immediately",
    "click link to reset password",
    "congratulations you won iphone",
    "claim your lottery prize now",
    "your bank account is locked",
    "verify your debit card details",
    "update your payment information",
    "your account will be suspended",
    "click to avoid account suspension",

    "free netflix subscription click here",
    "you won free amazon gift card",
    "click here to claim reward",
    "verify your paypal account now",
    "login here to secure account",
    "your account needs verification",
    "security alert login required",
    "unauthorized login detected verify now",
    "click here to secure your account",
    "claim your free coupon now",

    "bank alert verify your account",
    "reset your password immediately",
    "click link to avoid suspension",
    "confirm your account details",
    "your account has suspicious activity",
    "verify your identity now",
    "urgent login required",
    "click here to update account",
    "free reward waiting click now",
    "claim your prize today",

    "your account is temporarily locked",
    "verify your email now",
    "click here for free recharge",
    "urgent verify now",
    "account compromised click here",
    "confirm your password now",
    "secure your account immediately",
    "login required urgent",
    "verify your bank login",
    "claim free data now",

    "update your bank details now",
    "click here to claim cash prize",
    "you have won lottery click here",
    "account suspended login now",
    "verify your credentials immediately",
    "urgent password reset required",
    "security alert verify account",
    "click this link now urgent",
    "free offer click now",
    "verify your information now",

    "fakebank-login.xyz",
    "secure-login-alert.com",
    "verify-now-bank.com",
    "free-recharge-offer.xyz",
    "paypal-secure-login.xyz",
    "amazon-gift-free.xyz",
    "google-security-alert.xyz",
    "bank-login-update.xyz",
    "click-now-free.xyz",
    "account-verify-now.xyz",

    # ---------------- SAFE EXAMPLES ----------------

    "hello how are you",
    "let us meet tomorrow",
    "this is safe website",
    "google.com official site",
    "github.com official repository",
    "college assignment submission",
    "meeting scheduled tomorrow",
    "welcome to official portal",
    "this is normal message",
    "youtube.com official site",

    "thank you for your help",
    "please review the document",
    "class starts at 9am",
    "project meeting today",
    "this is official message",
    "check your email inbox",
    "normal conversation message",
    "let us complete assignment",
    "see you in college",
    "have a great day",

    "amazon.com official website",
    "linkedin.com official site",
    "microsoft.com official site",
    "apple.com official website",
    "facebook.com official site",
    "instagram.com official site",
    "twitter.com official site",
    "netflix.com official site",
    "college portal login",
    "student dashboard",

    "assignment submission today",
    "team meeting at 5pm",
    "please complete homework",
    "class test tomorrow",
    "project submission deadline",
    "normal safe link google.com",
    "safe educational website",
    "official college website",
    "secure login portal",
    "student learning platform",

    "hello friend good morning",
    "let us go to college",
    "this is safe message",
    "welcome to classroom",
    "official learning website",
    "normal safe content",
    "team collaboration platform",
    "official student portal",
    "course material available",
    "educational content website"
]

labels = [

    # phishing = 1
    1,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,1,1,1,
    1,1,1,1,1,1,1,1,1,1,

    # safe = 0
    0,0,0,0,0,0,0,0,0,0,
    0,0,0,0,0,0,0,0,0,0,
    0,0,0,0,0,0,0,0,0,0,
    0,0,0,0,0,0,0,0,0,0,
    0,0,0,0,0,0,0,0,0,0
]



# Training

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

model = LogisticRegression()
model.fit(X, labels)


# Keyword Detection

phishing_keywords = [
    "free", "win", "urgent", "verify", "claim",
    "password", "bank", "login", "reward",
    "suspended", "click", "security-alert"
]


def keyword_threat_score(text):

    score = 0

    for word in phishing_keywords:
        if word in text.lower():
            score += 10

    return score


# Stores the search history in CSV File

def store_history(user_input, threat_score, result):

    df = pd.read_csv(FILE)

    new_row = {
        "Time": datetime.now(),
        "Input": user_input,
        "Threat Score": str(threat_score) + "%",
        "Result": result
    }

    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    df.to_csv(FILE, index=False)


# UI

st.title("🛡️ Smart Digital Guardian")
st.subheader("AI-Based Cyber Threat Detection")

user_input = st.text_input("Enter Link or Message")

# Function after clicking analysis

if st.button("Analyze"):

    if user_input:

        input_vector = vectorizer.transform([user_input])
        probability = model.predict_proba(input_vector)[0][1]

        ai_score = probability * 100

        keyword_score = keyword_threat_score(user_input)

        threat_score = int((ai_score + keyword_score) / 2)

        if threat_score > 100:
            threat_score = 100


        st.subheader("Threat Score:")
        st.write(f"{threat_score}%")


        if threat_score >= 60:
            result = "Dangerous"
            st.error("⚠️ Dangerous / Phishing detected")

        elif threat_score >= 30:
            result = "Suspicious"
            st.warning("⚠️ Suspicious link")

        else:
            result = "Safe"
            st.success("✅ Safe content")


        # store search
        store_history(user_input, threat_score, result)

    else:
        st.warning("Please enter input")


# To view the search history

if st.button("View Search History"):

    df = pd.read_csv(FILE)

    st.subheader("Search History")

    st.dataframe(df)


Overwriting app.py


In [ ]:
!streamlit run app.py &>/content/logs.txt &


In [ ]:
!pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("33V5OAIuX4lRabueB4uvjX61xMm_7iDzoPxboEBQtWaNKmYio")


In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)

print(public_url)


NgrokTunnel: "https://excitably-overdecadent-shanna.ngrok-free.dev" -> "http://localhost:8501"
